In [ ]:
# Program 1: Uber Ride Fare Prediction
# Dataset: https://www.kaggle.com/datasets/yasserh/uber-fares-dataset (uber.csv)

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

warnings.filterwarnings('ignore')
%matplotlib inline

In [ ]:
# Load dataset
df = pd.read_csv('uber.csv')
print(f"Original shape: {df.shape}")
df.head()

In [ ]:
# Pre-processing & Feature Extraction
df = df.drop(['Unnamed: 0', 'key'], axis=1, errors='ignore')
df = df.dropna()

df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'], errors='coerce')
df = df.dropna(subset=['pickup_datetime'])
df['hour'] = df['pickup_datetime'].dt.hour
df['day'] = df['pickup_datetime'].dt.day
df['month'] = df['pickup_datetime'].dt.month
df['year'] = df['pickup_datetime'].dt.year
df['day_of_week'] = df['pickup_datetime'].dt.dayofweek
df = df.drop('pickup_datetime', axis=1)

# Haversine distance
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0)**2
    return 2.0 * R * np.arcsin(np.sqrt(a))

df['distance_km'] = haversine(
    df['pickup_latitude'], df['pickup_longitude'],
    df['dropoff_latitude'], df['dropoff_longitude']
)
df.head()

In [ ]:
# Outlier Removal
df = df[(df['fare_amount'] > 0) & (df['fare_amount'] < 200)]
df = df[(df['distance_km'] > 0) & (df['distance_km'] < 200)]
df = df[(df['passenger_count'] > 0) & (df['passenger_count'] <= 6)]
df = df[(df['pickup_latitude'].between(40.0, 42.0)) &
        (df['pickup_longitude'].between(-75.0, -72.0)) &
        (df['dropoff_latitude'].between(40.0, 42.0)) &
        (df['dropoff_longitude'].between(-75.0, -72.0))]
print(f"Cleaned shape: {df.shape}")

In [ ]:
# Correlation Heatmap
numeric_cols = df.select_dtypes(include=[np.number]).columns
plt.figure(figsize=(10, 8))
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
# Model Training & Evaluation
features = ['distance_km', 'passenger_count', 'hour', 'day', 'month', 'year', 'day_of_week']
X = df[features]
y = df['fare_amount']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)

# Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

print("Linear Regression:")
print(f"  R2 Score : {r2_score(y_test, lr_pred):.4f}")
print(f"  RMSE     : {np.sqrt(mean_squared_error(y_test, lr_pred)):.4f}")
print(f"  MAE      : {mean_absolute_error(y_test, lr_pred):.4f}")

print("\nRandom Forest:")
print(f"  R2 Score : {r2_score(y_test, rf_pred):.4f}")
print(f"  RMSE     : {np.sqrt(mean_squared_error(y_test, rf_pred)):.4f}")
print(f"  MAE      : {mean_absolute_error(y_test, rf_pred):.4f}")